# Compare flux of tracers across variable forcing period

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from min3p.output import read_min3p_sequence
from tqdm.notebook import tqdm
from byte_util import all_sites, states_per_site

all_scenarios = ['longterm', 'monthly', 'daily', 'hourly']

## Plot tracer breakthrough curves

In [ ]:
replot = True

for site in tqdm(all_sites):
    if not replot:
        continue
    fig, ax = plt.subplots(3, 3, figsize=(10, 7), sharex='all', tight_layout=True)

    for i, scenario in enumerate(all_scenarios):
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        sim_name = scenario

        gbt, gbt_cols, grid_cells = read_min3p_sequence(f'{sim_name}_1.gbt', folder=sim_folder, ftype='transient')
        # Iterate over tracers
        for j in range(3):
            # Iterate over control planes
            axi = 0
            for k, grid_cell in enumerate(grid_cells):
                if grid_cell in [101, 201, 301]:

                    ax[j, axi].plot(gbt[k, gbt_cols.index('time')],
                                    gbt[k, gbt_cols.index(f'psi{j+1:02d}')], lw=1, label=scenario)

                    # Add depth label to the subplot
                    if i == 0:
                        depth = f'{401-grid_cell} cm'
                        ax[j, axi].text(0.97, 0.92, depth, transform=ax[j, axi].transAxes, ha='right')

                    axi += 1

    mask = gbt[-1, gbt_cols.index(f'psi01')] > 0.01
    idx = np.argmax(np.flip(mask))
    limit = gbt[-1, 0, -idx]

    ax[0, 0].set(ylabel='Shallow tracer (M)', xlim=[0, limit*1.4])
    ax[1, 0].set(ylabel='Deep tracer (M)')
    ax[2, 0].set(ylabel='Alk tracer (M)')
    ax[0, 1].set(title='Shallow tracer (0-30 cm depth)')
    ax[1, 1].set(title='Deep tracer (30-100 cm depth)')
    ax[2, 1].set(title='Constant production tracer (0-30 cm depth)')

    title = f'{site} Series ({states_per_site[site]})'
    fig.suptitle(title, fontsize=16)

    for i in range(3):
        ax[2, i].set(xlabel='Time (d)')

    ax[0, 0].legend()

    fig.savefig(f'plots/{site}_TracerBreakthrough.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

## Calculate cumulative tracer flux across all scenarios

In [ ]:
from byte_util.met_transport import calc_tracer_flux

recalculate = False
tracers = ['psi01', 'psi02', 'psi03']
control_planes = [50, 100, 200, 300]  # Depth, in cm

for site in tqdm(all_sites):
    if not recalculate:
        continue
    for scenario in all_scenarios:
        sim_folder = Path(f'../min3p_runs/{site}/{scenario}')
        sim_name = scenario

        df = calc_tracer_flux(sim_folder, sim_name, tracers=tracers)

        # Calculate cumulative flux over time, correcting for "background" flux
        for cp in control_planes:
            for tracer in tracers:
                dt = np.diff(df['time'].values)
                dt = np.insert(dt, 0, df.loc[0, 'time'])
                # Estimate background flux as q*background_conc (0.001 mol/L)
                back_flux = -df[f'q_{cp}cm_m.d']*(0.001*1000)
                net_flux = df[f'{tracer}_flux_{cp}cm_mol.m2.d'] - back_flux
                df[f'{tracer}_cumulative_{cp}cm_mol.m2'] = np.cumsum(dt*net_flux)
        df.to_csv(f'output_data/{site}_{scenario}_TracerFlux.csv', index=False)

In [ ]:
# Plot shallow tracer flux over time
replot = True

for site in tqdm(all_sites):
    if not replot:
        continue
    fig, ax = plt.subplots(2, 2, figsize=(8, 6), sharex='all', sharey='all', tight_layout=True)

    for i, cp in enumerate(control_planes):
        for j, scenario in enumerate(all_scenarios):
            df = pd.read_csv(f'output_data/{site}_{scenario}_TracerFlux.csv')
            ax.flatten()[i].plot(df['time'], df[f'psi01_cumulative_{cp}cm_mol.m2']*1000,
                                 lw=1, label=scenario)
            # Add depth label to the subplot
            if j == 0:
                depth = f'{cp} cm'
                ax.flatten()[i].text(0.02, 0.92, depth, ha='left',
                                     transform=ax.flatten()[i].transAxes)

    for i in range(2):
        ax[i, 0].set(ylabel='Tracer flux (mmol/m$^2$)')
        ax[1, i].set(xlabel='Time (d)', xlim=[0, 3650])

    ax[0, 0].legend()
    fig.suptitle(f'{site} Series ({states_per_site[site]}): Shallow tracer (0-30 cm depth)')

    fig.savefig(f'plots/{site}_ShallowTracerFlux.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

In [ ]:
# Plot deep tracer flux over time
replot = True

for site in tqdm(all_sites):
    if not replot:
        continue
    fig, ax = plt.subplots(2, figsize=(6, 6), sharex='all', sharey='all', tight_layout=True)

    for i, cp in enumerate(control_planes[-2:]):
        for j, scenario in enumerate(all_scenarios):
            df = pd.read_csv(f'output_data/{site}_{scenario}_TracerFlux.csv')
            ax[i].plot(df['time'], df[f'psi02_cumulative_{cp}cm_mol.m2']*1000,
                       lw=1, label=scenario)
            # Add depth label to the subplot
            if j == 0:
                ax[i].text(0.02, 0.92, f'{cp} cm', ha='left', transform=ax[i].transAxes)

    for i in range(2):
        ax[i].set(ylabel='Tracer flux (mmol/m$^2$)')
    ax[1].set(xlabel='Time (d)', xlim=[0, 3650])

    ax[0].legend()
    fig.suptitle(f'{site} Series ({states_per_site[site]}): Deep tracer (30-100 cm depth)')

    fig.savefig(f'plots/{site}_DeepTracerFlux.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

In [ ]:
# Plot constant production tracer flux over time
replot = True

for site in tqdm(all_sites):
    if not replot:
        continue
    fig, ax = plt.subplots(2, figsize=(6, 6), sharex='all', sharey='all', tight_layout=True)

    for i, cp in enumerate(control_planes[-2:]):
        for j, scenario in enumerate(all_scenarios):
            df = pd.read_csv(f'output_data/{site}_{scenario}_TracerFlux.csv')
            ax[i].plot(df['time'], df[f'psi03_cumulative_{cp}cm_mol.m2']*1000,
                       lw=1, label=scenario)
            # Add depth label to the subplot
            if j == 0:
                ax[i].text(0.02, 0.92, f'{cp} cm', ha='left', transform=ax[i].transAxes)

    for i in range(2):
        ax[i].set(ylabel='Tracer flux (mmol/m$^2$)')
    ax[1].set(xlabel='Time (d)', xlim=[0, 3650])

    ax[0].legend()
    fig.suptitle(f'{site} Series ({states_per_site[site]}): Alk tracer (0-30 cm production)')

    fig.savefig(f'plots/{site}_CumulativeAlkalinityTracerFlux.png', dpi=300, bbox_inches='tight')

    if site != 'Cecil':
        plt.close(fig)

## Calculate tracer breakthrough and flux statistics for all simulations starting 2001-01-01

In [ ]:
stats = pd.DataFrame(columns=['site', 'scenario', 'tracer', 'control_plane', 'peak_time'])
stats.set_index(['site', 'scenario', 'tracer', 'control_plane'], inplace=True)

for site in all_sites:
    for scenario in all_scenarios:
        df = pd.read_csv(f'output_data/{site}_{scenario}_TracerFlux.csv')

        cp_str = [col.split('cm')[0].split('_')[-1] for col in df.columns if 'time' not in col]
        control_planes = list({int(cp) for cp in cp_str})
        control_planes.sort()
        tracers = list(set([col.split('_')[0] for col in df.columns if 'conc' in col]))

        for tracer in tracers:
            for cp in control_planes:
                idx = df[f'{tracer}_conc_{cp}cm_mol.L'].argmax()
                peak_conc_time = df.loc[idx, 'time']
                stats.loc[(site, scenario, tracer, cp), 'peak_time'] = peak_conc_time
print(stats)

In [ ]:
tracer_names = {'psi01': 'Shallow tracer (0-30 cm)',
                'psi02': 'Deep tracer (30-100 cm)'}
err_scenarios = ['longterm', 'monthly', 'daily']

tracer = 'psi01'
cp = 300
for tracer in ['psi01', 'psi02']:
    for cp in control_planes:
        sub_df = stats.loc[(slice(None), slice(None), tracer, cp), 'peak_time'].unstack('scenario')
        sub_df = sub_df.droplevel([1, 2])
        err = sub_df[err_scenarios].sub(sub_df['hourly'], axis=0).div(sub_df['hourly'], axis=0) * 100

        x = np.arange(len(err))
        w = 0.24

        fig, ax = plt.subplots(figsize=(10, 5), tight_layout=True)

        for i, scenario in enumerate(err_scenarios):
            ax.bar(x + (i - 1) * w, err[scenario], width=w, label=scenario)

        ax.axhline(0, color='k', lw=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(err.index, rotation=45, ha='right')
        ax.set_ylabel('Percent error in peak arrival time (%)')
        tracer_name = tracer_names[tracer]
        ax.set_title(f'{tracer_name} at {cp} cm depth')
        ax.legend(title='Scenario')

        tracer_save_name = tracer_name.split(' ')[0]
        fig.savefig(f'plots/{tracer_save_name}Tracer_{cp}cm_peak_tracer_percent_error.png',
                    dpi=300, bbox_inches='tight')

        if tracer != 'psi01' or cp != 300:
            plt.close(fig)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, tight_layout=True)
w = 0.22
colors = ['peachpuff', 'orange', 'tomato']

for ax, tracer in zip(axes, ['psi01', 'psi02']):
    x = stats.xs(tracer, level='tracer')['peak_time'].unstack('scenario')
    err = x[err_scenarios].sub(x['hourly'], axis=0).div(x['hourly'], axis=0) * 100

    data, pos = [], []
    for i, cp in enumerate(control_planes):
        for j, scenario in enumerate(err_scenarios):
            data.append(err.loc[(slice(None), cp), scenario].dropna())
            pos.append(i + (j - 1) * w)

    bplot = ax.boxplot(data, positions=pos, widths=0.18, patch_artist=True)
    for patch, color in zip(bplot['boxes'], colors*4):
        patch.set_facecolor(color)
    ax.axhline(0, color='k', lw=0.8)
    ax.set(
        xticks=np.arange(len(control_planes)),
        xticklabels=[f'{cp} cm' for cp in control_planes],
        xlabel='Control plane',
        ylabel='Percent error in peak arrival time (%)',
        title=f'{tracer_names[tracer]} relative to hourly',
    )

axes[1].set(ylabel='')
fig.legend(err_scenarios, bbox_to_anchor=(0.98, 0.3), bbox_transform=fig.transFigure)
fig.suptitle('Only for simulations starting 2001-01-01')
fig.savefig('plots/PeakTracerPercentError_Boxplots.png', dpi=300, bbox_inches='tight')

## Create boxplot of tracer breakthrough statistics for simulations with variable start dates

In [ ]:
# Create copy of `stats` df from above (from all simulations starting 2001-01-01) as a reference
ref_stats = stats.copy()

csv_paths = sorted(Path('output_data/variable_start_peak_tracer').glob('*TracerBreakthrough.csv'))

dfs = []
for path in csv_paths:
    start_date = path.stem[:10]
    df = pd.read_csv(path, index_col=['site', 'scenario', 'tracer', 'control_plane'])

    # Add longterm simulations from ref_stats
    df = pd.concat((df, ref_stats.loc[(slice(None), 'longterm'), :]), axis=0)

    df['start_date'] = pd.to_datetime(start_date)
    dfs.append(df.set_index('start_date', append=True))

stats_all = pd.concat(dfs).reorder_levels(
    ['start_date', 'site', 'scenario', 'tracer', 'control_plane']
).sort_index()

In [ ]:
tracer = 'psi01'
cp = 100

for cp in control_planes:
    for tracer in ['psi01', 'psi02']:
        if tracer == 'psi02' and cp < 200:
            continue
        x = (stats_all.xs((tracer, cp),
                          level=('tracer', 'control_plane'))['peak_time'].unstack('scenario'))

        pct_err = x[err_scenarios].sub(x['hourly'], axis=0).div(x['hourly'], axis=0) * 100
        abs_err = x[err_scenarios].sub(x['hourly'], axis=0)
        site_order = pct_err.index.get_level_values('site').unique()

        fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True, tight_layout=True)
        w = 0.22
        ylabels = ['Error in peak arrival time (d)', 'Percent error in peak arrival time (%)']

        for ax, err, ylabel in zip(axes, [abs_err, pct_err], ylabels):
            data, pos = [], []

            for i, site in enumerate(site_order):
                for j, scenario in enumerate(err_scenarios):
                    data.append(err.loc[(slice(None), site), scenario].dropna())
                    pos.append(i + (j - 1) * w)

            bplot = ax.boxplot(data, positions=pos, widths=0.18, patch_artist=True)

            for patch, color in zip(bplot['boxes'], colors * len(site_order)):
                patch.set_facecolor(color)

            # Add n to upper left
            ax.text(0.01, 0.98, f'n={len(csv_paths):d}', va='top', ha='left', transform=ax.transAxes)

            ax.axhline(0, color='k', lw=0.8)
            ax.set(xticks=np.arange(len(site_order)), xticklabels=site_order, ylabel=ylabel,
                   title=f'{tracer_names[tracer]} at {cp} cm control plane')

        fig.legend(err_scenarios, bbox_to_anchor=(0.98, 0.47), bbox_transform=fig.transFigure)
        tracer_depth = tracer_names[tracer].split(' ')[0]

        fig.savefig(f'plots/variable_start/{tracer_depth}Tracer_{cp}cm_PeakTracerError_Boxplots.png',
                    dpi=300, bbox_inches='tight')
        if tracer != 'psi01' or cp != 300:
            plt.close(fig)

## Perform DGSA

In [ ]:
# Choose control plane and tracer
tracer = 'psi01'
cp = 100

# Get soil parameters for DGSA
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

soil_params_path = (f'{s3_base_path}/'
                    'input-data/processed-data/soil_physical_parameters.parquet')
soil_params = pd.read_parquet(soil_params_path)

# Add soil params to results
x = (stats_all.xs((tracer, cp),
                  level=('tracer', 'control_plane'))['peak_time'].unstack('scenario'))

results = x[err_scenarios].sub(x['hourly'], axis=0).div(x['hourly'], axis=0) * 100
results.reset_index(inplace=True)

# Add parameter columns to results
param_cols = [c for c in soil_params.columns if c not in ('top_m', 'bottom_m')]

for col in param_cols:
    results[col] = np.nan
    for site in all_sites:
        sp_mask = (soil_params.loc[site, 'top_m'] < cp/100) * (soil_params.loc[site, 'bottom_m'] > cp/100)
        result_mask = results['site'] == site
        results.loc[result_mask, col] = soil_params.loc[site, col][sp_mask].values[0]

# Get day of year and year from start_date
results['doy'] = results['start_date'].dt.day_of_year
results['year'] = results['start_date'].dt.year

# Create numeric site column for dgsa
site_no = {site: i for i, site in enumerate(all_sites)}
results['site_no'] = results['site'].map(site_no)

In [ ]:
from pyDGSA.dgsa import dgsa
from pyDGSA.plot import vert_pareto_plot

scenario = 'longterm'

# Construct labels by pct_error < -10%, pct_error > 10%, and -10% <= pct_error <= 10%
# Accurate == 1, overestimation == 2, underestimation == 0
labels = np.where(results[scenario] < -10, 0,
                  np.where(results[scenario] > 10, 2, 1))

dgsa_params = param_cols + ['site_no', 'doy', 'year']
sens = dgsa(results[dgsa_params].values, labels, parameter_names=dgsa_params, confidence=True)
fig, ax = vert_pareto_plot(sens, confidence=True)

ax.set(title=f'{scenario}: {tracer} at {cp} cm control plane')

In [ ]:
# Scatter plot of arrival time error vs day of year
fig, ax = plt.subplots(figsize=(8, 6), tight_layout=True)
for scenario in err_scenarios:
    ax.scatter(results['doy'], results[scenario], label=scenario, alpha=0.5)
ax.axhline(0, color='k', lw=0.8)
ax.set(xlabel='Day of year', ylabel='Percent error in peak arrival time (%)', title=f'{tracer_names[tracer]} at {cp} cm control plane')

# Add Kcb (crop coefficients) to the plot
growth_stages = {
    'Lini_crop': 30,
    'Ldev': 40,
    'Lmid': 50,
    'Lend': 50,
}
kcb = {
    'ini': 0.15,
    'mid': 1.15,
    'end': 0.15,
}
d0 = 0
d1 = 105  # Planting on April 15
d2 = d1 + growth_stages['Lini_crop']
d3 = d2 + growth_stages['Ldev']
d4 = d3 + growth_stages['Lmid']
d5 = d4 + growth_stages['Lend']
d6 = 365

ax1 = ax.twinx()
kcb_x = (d0, d1, d1, d2, d3, d4, d5, d5, d6)
kbv_y = (0, 0, kcb['ini'], kcb['ini'], kcb['mid'], kcb['mid'], kcb['end'], 0, 0)
ax1.plot(kcb_x, kbv_y, color='0.3', ls='--')
ax.legend()